# PatchTST-Lite Only — Frozen Original IJIES Protocol

This notebook is derived from the **original pre-PatchTST experiment notebook**.

**Purpose:** add only the PatchTST-lite comparator requested in the second-round IJIES review, while keeping all previously reported LSTM/GRU/baseline results frozen.

Important rules implemented here:
- Existing LSTM/GRU/GRU-N-Beats/CNN-GRU-Attention models are **not retrained**.
- The original dataset preparation, 24-step input window, horizons, feature engineering, chronological split, rolling-origin folds, scaling procedure, seeds, epochs, batch size, and early-stopping budget are preserved.
- PatchTST-lite uses the same **58 Raw sensor + temporal features** as Raw LSTM/Raw GRU and the literature baselines.
- Fold 0 uses five seeds: 42, 7, 123, 2024, 99.
- Rolling-origin folds 1–2 use seed 42 only, matching the manuscript's stated uncertainty protocol.
- No hyperparameter re-search is performed for the existing models.
- PatchTST-lite architecture is fixed in this notebook and is not used to alter the previously published model-selection protocol.

Run `QUICK_TEST_MODE=True` once as a smoke test. Then set it to `False` and rerun from the top for final PatchTST-only results.


In [ ]:
# =========================
# 0. Setup & Global Config
# =========================
import os, json, zipfile, random, warnings, time, itertools
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              precision_score, recall_score, f1_score, confusion_matrix,
                              roc_auc_score, average_precision_score)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# ---- Master switches -------------------------------------------------
QUICK_TEST_MODE = False   # <<< set False for the full run once this passes cleanly
SEEDS_FULL = [42, 7, 123, 2024, 99]        # 5 seeds for the full run (Reviewer 2 #5)
SEEDS_QUICK = [42]                         # quick sanity run
SEEDS = SEEDS_QUICK if QUICK_TEST_MODE else SEEDS_FULL

WINDOW = 24
HORIZONS = [1, 2, 3, 4]
HORIZON_LABELS = {1: '15 min', 2: '30 min', 3: '45 min', 4: '60 min'}
INTERVAL_MINUTES = 15

EPOCHS = 5 if QUICK_TEST_MODE else 120
BATCH_SIZE = 32
PATIENCE = 3 if QUICK_TEST_MODE else 15

N_BLOCKS = 5           # number of chronological blocks per pond used for walk-forward folds
FOLD_IDS_QUICK = [0]   # only the original split, for a fast smoke test
FOLD_IDS_FULL = [0, 1, 2]  # 0 = original 70/15/15 split, 1-2 = rolling-origin walk-forward folds

RESULT_DIR = 'patchtst_only_frozen_results'
os.makedirs(RESULT_DIR, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('QUICK_TEST_MODE:', QUICK_TEST_MODE)
print('SEEDS used this run:', SEEDS)
print('RESULT_DIR:', RESULT_DIR)
print('MODE: PATCHTST-LITE ONLY; existing published model results remain frozen')


## 1. Upload / Read `iot_data.csv`

In [ ]:
# =========================
# 1. Upload / read iot_data.csv
# =========================
DATA_PATH = 'iot_data.csv'

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print('File iot_data.csv belum ditemukan. Silakan upload iot_data.csv')
        uploaded = files.upload()
        if DATA_PATH not in uploaded:
            first_file = list(uploaded.keys())[0]
            os.rename(first_file, DATA_PATH)
            print(f'File {first_file} diganti nama menjadi {DATA_PATH}')
    except Exception as e:
        raise FileNotFoundError('Upload iot_data.csv terlebih dahulu ke runtime Colab.') from e

def read_iot_csv(path):
    try:
        df0 = pd.read_csv(path, sep=None, engine='python')
    except Exception:
        df0 = pd.read_csv(path, sep=';')
    if len(df0.columns) == 1 and ';' in df0.columns[0]:
        df0 = pd.read_csv(path, sep=';')
    return df0

raw_df = read_iot_csv(DATA_PATH)
print('Kolom awal:', raw_df.columns.tolist())
print('Shape awal:', raw_df.shape)
display(raw_df.head())


## 2. Clean & Normalize Data

In [ ]:
# =========================
# 2. Normalisasi kolom, timestamp, nama kolam, dan numeric columns
# =========================
df = raw_df.copy()

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

rename_map = {
    'timestamp': 'timestamp', 'time': 'timestamp', 'datetime': 'timestamp', 'waktu': 'timestamp', 'tanggal': 'timestamp',
    'pond': 'pond', 'kolam': 'pond', 'pool': 'pond', 'jenis_kolam': 'pond',
    'temp': 'temp', 'temperature': 'temp', 'suhu': 'temp',
    'ph': 'ph', 'p_h': 'ph',
    'do': 'do', 'dissolved_oxygen': 'do', 'oxygen': 'do', 'dissolvedoxygen': 'do'
}
df = df.rename(columns=rename_map)

required_cols = ['timestamp', 'pond', 'temp', 'ph', 'do']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print('Kolom setelah normalisasi:', df.columns.tolist())
    raise ValueError(f'Kolom wajib belum ditemukan: {missing}. Cek header iot_data.csv atau tambah rename_map.')

df = df[required_cols].copy()

# Bersihkan timestamp: contoh 01/01/26 00.00 -> 01/01/26 00:00
s = df['timestamp'].astype(str).str.strip()
s = s.str.replace(r'(\d{1,2})\.(\d{2})(:\d{2})?$', lambda m: m.group(0).replace('.', ':'), regex=True)

dt = pd.to_datetime(s, format='%d/%m/%y %H:%M', errors='coerce')
if dt.isna().sum() > 0:
    dt2 = pd.to_datetime(s, dayfirst=True, errors='coerce')
    dt = dt.fillna(dt2)
df['timestamp'] = dt

failed_timestamp = df['timestamp'].isna().sum()
print('Jumlah timestamp gagal parse:', failed_timestamp)
if failed_timestamp > 0:
    display(df[df['timestamp'].isna()].head(20))
    raise ValueError('Masih ada timestamp yang gagal dibaca. Cek format tanggal pada baris yang ditampilkan.')

def to_numeric_clean(series):
    return pd.to_numeric(series.astype(str).str.strip().str.replace(',', '.', regex=False), errors='coerce')

df['temp'] = to_numeric_clean(df['temp'])
df['ph'] = to_numeric_clean(df['ph'])
df['do'] = to_numeric_clean(df['do'])

# NOTE: nama kolam DIPERTAHANKAN dalam Bahasa Inggris (Catfish/Gourami/Nilem/Tilapia)
# agar konsisten dengan manuskrip dan tidak perlu mapping balik saat menulis Bab 4/paper.
df['pond'] = df['pond'].astype(str).str.strip()

df = df.sort_values(['pond', 'timestamp']).reset_index(drop=True)

print('Kolom setelah normalisasi:', df.columns.tolist())
print('Nama kolam:', sorted(df['pond'].unique().tolist()))
print('Shape:', df.shape)

# Tampilkan rentang kalender tiap pond -- PENTING: pond-pond ini TIDAK diobservasi
# paralel, melainkan berurutan secara waktu. Ini perlu disebutkan eksplisit di manuskrip.
calendar_span = df.groupby('pond')['timestamp'].agg(['min', 'max'])
print('\nRentang kalender tiap pond (pond diobservasi berurutan, bukan paralel):')
display(calendar_span)
calendar_span.to_csv(f'{RESULT_DIR}/pond_calendar_span.csv')

display(df.head())


## 3. Data Readiness Checks

In [ ]:
# =========================
# 3. Data readiness checks dan physical range filtering
# =========================
readiness_summary = []

n_initial = len(df)
readiness_summary.append({'step': 'initial_rows', 'rows': n_initial})

dup_count = df.duplicated(subset=['pond', 'timestamp']).sum()
readiness_summary.append({'step': 'duplicated_pond_timestamp', 'rows': int(dup_count)})
df = df.drop_duplicates(subset=['pond', 'timestamp'], keep='first').copy()

invalid_mask = (
    (df['temp'] < 0) | (df['temp'] > 50) |
    (df['ph'] < 0) | (df['ph'] > 14) |
    (df['do'] < 0) | (df['do'] > 20) |
    df[['temp', 'ph', 'do']].isna().any(axis=1)
)
readiness_summary.append({'step': 'invalid_or_missing_sensor_rows', 'rows': int(invalid_mask.sum())})
df_clean = df.loc[~invalid_mask].copy().reset_index(drop=True)
readiness_summary.append({'step': 'final_clean_rows', 'rows': len(df_clean)})

interval_summary = []
for pond, g in df_clean.groupby('pond'):
    g = g.sort_values('timestamp')
    diffs = g['timestamp'].diff().dropna().dt.total_seconds() / 60
    interval_summary.append({
        'pond': pond,
        'observations': len(g),
        'start_time': g['timestamp'].min(),
        'end_time': g['timestamp'].max(),
        'median_interval_minutes': diffs.median() if len(diffs) > 0 else np.nan,
        'non_15min_intervals': int((diffs != 15).sum()) if len(diffs) > 0 else 0
    })

readiness_df = pd.DataFrame(readiness_summary)
interval_df = pd.DataFrame(interval_summary)
readiness_df.to_csv(f'{RESULT_DIR}/data_readiness_summary.csv', index=False)
interval_df.to_csv(f'{RESULT_DIR}/interval_summary.csv', index=False)

df = df_clean
print('Data readiness summary')
display(readiness_df)
print('Interval summary')
display(interval_df)


## 4. Temporal Assessment (Suitability / Excursion Scoring)

In [ ]:
# =========================
# 4. Rentang acuan dan assessment temporal
# =========================
# Rentang species-specific TIDAK diubah dari manuskrip -- hanya key pond disamakan
# dengan nama Inggris yang dipakai konsisten di seluruh notebook & manuskrip.
RANGES = {
    'Catfish': {'temp_opt': (25, 30), 'temp_ops': (23, 32), 'ph_opt': (7.0, 8.5), 'ph_ops': (6.5, 9.0), 'do_opt': 3.0, 'do_ops': 2.0},
    'Gourami': {'temp_opt': (25, 30), 'temp_ops': (23, 32), 'ph_opt': (6.5, 8.5), 'ph_ops': (6.0, 9.0), 'do_opt': 5.0, 'do_ops': 4.0},
    'Nilem':   {'temp_opt': (26, 29), 'temp_ops': (24, 31), 'ph_opt': (6.5, 8.0), 'ph_ops': (6.0, 8.5), 'do_opt': 5.0, 'do_ops': 4.0},
    'Tilapia': {'temp_opt': (25, 30), 'temp_ops': (23, 32), 'ph_opt': (6.5, 8.5), 'ph_ops': (6.0, 9.0), 'do_opt': 5.0, 'do_ops': 4.0},
}

unknown_ponds = sorted(set(df['pond'].unique()) - set(RANGES.keys()))
if unknown_ponds:
    raise ValueError(f'Nama kolam belum ada di RANGES: {unknown_ponds}. Tambahkan mapping pond atau RANGES.')

def score_interval(value, opt_range, ops_range):
    opt_low, opt_high = opt_range
    ops_low, ops_high = ops_range
    if pd.isna(value):
        return np.nan
    if opt_low <= value <= opt_high:
        return 3
    if ops_low <= value <= ops_high:
        return 2
    return 1

def score_do(value, opt_min, ops_min):
    if pd.isna(value):
        return np.nan
    if value >= opt_min:
        return 3
    if value >= ops_min:
        return 2
    return 1

df['temp_score'] = np.nan
df['ph_score'] = np.nan
df['do_score'] = np.nan
df['do_ops_thr'] = np.nan
df['do_opt_thr'] = np.nan

for pond, idx in df.groupby('pond').groups.items():
    r = RANGES[pond]
    df.loc[idx, 'temp_score'] = df.loc[idx, 'temp'].apply(lambda v: score_interval(v, r['temp_opt'], r['temp_ops']))
    df.loc[idx, 'ph_score'] = df.loc[idx, 'ph'].apply(lambda v: score_interval(v, r['ph_opt'], r['ph_ops']))
    df.loc[idx, 'do_score'] = df.loc[idx, 'do'].apply(lambda v: score_do(v, r['do_opt'], r['do_ops']))
    df.loc[idx, 'do_ops_thr'] = r['do_ops']
    df.loc[idx, 'do_opt_thr'] = r['do_opt']

df['pond_score'] = df[['temp_score', 'ph_score', 'do_score']].mean(axis=1)

# Fitur assessment DO
df['do_excursion_flag'] = (df['do'] < df['do_ops_thr']).astype(int)
df['do_warning_flag'] = ((df['do'] < df['do_opt_thr']) & (df['do'] >= df['do_ops_thr'])).astype(int)
df['do_gap_ops'] = df['do'] - df['do_ops_thr']
df['do_gap_opt'] = df['do'] - df['do_opt_thr']

df['do_excursion_duration_steps'] = 0
for pond, g in df.groupby('pond'):
    duration = []
    cur = 0
    for flag in g['do_excursion_flag'].astype(int).values:
        cur = cur + 1 if flag == 1 else 0
        duration.append(cur)
    df.loc[g.index, 'do_excursion_duration_steps'] = duration

df['do_excursion_duration_minutes'] = df['do_excursion_duration_steps'] * INTERVAL_MINUTES

df.to_csv(f'{RESULT_DIR}/processed_iot_dataset_with_assessment.csv', index=False)

print('Assessment columns created:')
print([c for c in df.columns if 'score' in c or 'do_' in c])
display(df.head())


## 5. Assessment Summaries & Class-Prevalence Reporting (Reviewer 2 #6)

In [ ]:
# =========================
# 5. Assessment result summaries + class-prevalence reporting (Reviewer 2 #6)
# =========================
suitability_summary = df.groupby('pond').agg(
    obs=('do', 'size'),
    temp_score=('temp_score', 'mean'),
    ph_score=('ph_score', 'mean'),
    do_score=('do_score', 'mean'),
    pond_score=('pond_score', 'mean'),
    mean_temp=('temp', 'mean'),
    mean_ph=('ph', 'mean'),
    mean_do=('do', 'mean')
).reset_index()
suitability_summary.to_csv(f'{RESULT_DIR}/assessment_suitability_summary.csv', index=False)

def excursion_events_for_series(flags):
    events = []
    start = None
    length = 0
    vals = list(flags.astype(int).values)
    for i, flag in enumerate(vals):
        if flag == 1 and start is None:
            start = i
            length = 1
        elif flag == 1:
            length += 1
        elif flag == 0 and start is not None:
            events.append(length)
            start = None
            length = 0
    if start is not None:
        events.append(length)
    return events

exc_rows = []
for pond, g in df.groupby('pond'):
    events = excursion_events_for_series(g['do_excursion_flag'])
    durations = [e * INTERVAL_MINUTES / 60 for e in events]
    exc_rows.append({
        'pond': pond,
        'event_count': len(events),
        'total_duration_hours': float(np.sum(durations)) if durations else 0.0,
        'max_duration_hours': float(np.max(durations)) if durations else 0.0,
        'excursion_rate': float(g['do_excursion_flag'].mean()),
        # NEW -- explicit class-prevalence numbers requested by Reviewer 2 #6
        'n_timestamps': len(g),
        'n_low_do_timestamps': int(g['do_excursion_flag'].sum()),
        'positive_prevalence_pct': round(100 * g['do_excursion_flag'].mean(), 2),
    })
excursion_summary = pd.DataFrame(exc_rows)
excursion_summary.to_csv(f'{RESULT_DIR}/assessment_excursion_do_summary_with_prevalence.csv', index=False)

stability_summary = df.groupby('pond').agg(
    temp_mean=('temp', 'mean'), temp_sd=('temp', 'std'),
    ph_mean=('ph', 'mean'), ph_sd=('ph', 'std'),
    do_mean=('do', 'mean'), do_sd=('do', 'std')
).reset_index()
for param in ['temp', 'ph', 'do']:
    stability_summary[f'{param}_cv'] = stability_summary[f'{param}_sd'] / stability_summary[f'{param}_mean']
stability_summary.to_csv(f'{RESULT_DIR}/assessment_temporal_stability_summary.csv', index=False)

print('Suitability summary')
display(suitability_summary)
print('Excursion / DO event summary (with class prevalence -- Reviewer 2 #6)')
display(excursion_summary)
print('Stability summary')
display(stability_summary)


## 6. Feature Engineering + Explicit Formula Export (Reviewer 2 #7)

In [ ]:
# =========================
# 6. Feature engineering: temporal, suitability, excursion, stability
# =========================
df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60.0
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_index'] = (df['timestamp'] - df.groupby('pond')['timestamp'].transform('min')).dt.total_seconds() / (24 * 3600)

pond_dummies = pd.get_dummies(df['pond'], prefix='pond', dtype=int)
df = pd.concat([df, pond_dummies], axis=1)
POND_FEATURES = pond_dummies.columns.tolist()

LAGS = [1, 2, 3, 4, 8, 12, 24]
WINDOWS = [4, 8, 24]

new_cols = {}
for pond, g in df.groupby('pond'):
    idx = g.index
    for col in ['temp', 'ph', 'do']:
        for lag in LAGS:
            new_cols[f'{col}_lag{lag}'] = new_cols.get(f'{col}_lag{lag}', pd.Series(index=df.index, dtype=float))
            new_cols[f'{col}_lag{lag}'].loc[idx] = g[col].shift(lag)
        for win in WINDOWS:
            for suffix, fn in [('roll_mean', 'mean'), ('roll_sd', 'std'), ('roll_min', 'min'), ('roll_max', 'max')]:
                key = f'{col}_{suffix}{win}'
                new_cols[key] = new_cols.get(key, pd.Series(index=df.index, dtype=float))
                new_cols[key].loc[idx] = getattr(g[col].rolling(win, min_periods=win), fn)()
            cv_key = f'{col}_roll_cv{win}'
            new_cols[cv_key] = new_cols.get(cv_key, pd.Series(index=df.index, dtype=float))
            new_cols[cv_key].loc[idx] = new_cols[f'{col}_roll_sd{win}'].loc[idx] / new_cols[f'{col}_roll_mean{win}'].loc[idx]
    for col in ['temp_score', 'ph_score', 'do_score', 'pond_score', 'do_excursion_flag', 'do_warning_flag']:
        for win in WINDOWS:
            mean_key = f'{col}_roll_mean{win}'
            sum_key = f'{col}_roll_sum{win}'
            new_cols[mean_key] = new_cols.get(mean_key, pd.Series(index=df.index, dtype=float))
            new_cols[sum_key] = new_cols.get(sum_key, pd.Series(index=df.index, dtype=float))
            new_cols[mean_key].loc[idx] = g[col].rolling(win, min_periods=win).mean()
            new_cols[sum_key].loc[idx] = g[col].rolling(win, min_periods=win).sum()

df = pd.concat([df, pd.DataFrame(new_cols)], axis=1)

BASE_FEATURES = ['temp', 'ph', 'do', 'hour_sin', 'hour_cos', 'day_index'] + POND_FEATURES

# -------------------------------------------------------------------
# FIXED (previously buggy): TEMPORAL_FEATURES is now built EXPLICITLY from
# the sensor lag/rolling columns we actually generated, instead of guessing
# via string matching. The old regex-based filter --
#   c.split('_')[0] in ['temp','ph','do']
# -- incorrectly matched columns like "temp_score_roll_mean4" and
# "do_excursion_flag_roll_mean8" too, because their FIRST token also happens
# to be "temp"/"do". That silently leaked 15 suitability/excursion-derived
# columns into every "Raw" and "SOTA baseline" feature set, contaminating
# the ablation study's control group. This explicit construction cannot
# make that mistake because it only ever appends the exact names created
# in the lag/rolling loop above.
# -------------------------------------------------------------------
TEMPORAL_FEATURES = []
for _col in ['temp', 'ph', 'do']:
    for _lag in LAGS:
        TEMPORAL_FEATURES.append(f'{_col}_lag{_lag}')
    for _win in WINDOWS:
        TEMPORAL_FEATURES.append(f'{_col}_roll_mean{_win}')
        TEMPORAL_FEATURES.append(f'{_col}_roll_min{_win}')
        TEMPORAL_FEATURES.append(f'{_col}_roll_max{_win}')
_leaked_check = [c for c in TEMPORAL_FEATURES if 'score' in c or 'flag' in c]
assert len(_leaked_check) == 0, f'TEMPORAL_FEATURES leaked non-sensor columns: {_leaked_check}'

SUITABILITY_FEATURES = ['temp_score', 'ph_score', 'do_score', 'pond_score', 'do_gap_ops', 'do_gap_opt'] + \
    [c for c in df.columns if any(c.startswith(p) for p in ['temp_score_roll', 'ph_score_roll', 'do_score_roll', 'pond_score_roll'])]

EXCURSION_FEATURES = ['do_excursion_flag', 'do_warning_flag', 'do_excursion_duration_steps', 'do_excursion_duration_minutes'] + \
    [c for c in df.columns if c.startswith('do_excursion_flag_roll') or c.startswith('do_warning_flag_roll')]

STABILITY_FEATURES = [c for c in df.columns if any(c.endswith(f'_roll_sd{w}') or c.endswith(f'_roll_cv{w}') for w in WINDOWS)]

def uniq(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

BASE_FEATURES = uniq(BASE_FEATURES)
TEMPORAL_FEATURES = uniq(TEMPORAL_FEATURES)
SUITABILITY_FEATURES = uniq(SUITABILITY_FEATURES)
EXCURSION_FEATURES = uniq(EXCURSION_FEATURES)
STABILITY_FEATURES = uniq(STABILITY_FEATURES)

feature_groups = {
    'BASE_FEATURES': BASE_FEATURES,
    'TEMPORAL_FEATURES': TEMPORAL_FEATURES,
    'SUITABILITY_FEATURES': SUITABILITY_FEATURES,
    'EXCURSION_FEATURES': EXCURSION_FEATURES,
    'STABILITY_FEATURES': STABILITY_FEATURES,
}
with open(f'{RESULT_DIR}/feature_groups.json', 'w') as f:
    json.dump(feature_groups, f, indent=2)

df.to_csv(f'{RESULT_DIR}/processed_iot_dataset_with_features.csv', index=False)

for name, feats in feature_groups.items():
    print(name, len(feats))

# -------------------------------------------------------------------
# NEW: explicit, human-readable formulas for every engineered feature
# group, exported for the manuscript's reproducibility section and
# for the response letter to Reviewer 2 #7.
# -------------------------------------------------------------------
feature_engineering_config = {
    "window_and_horizons": {
        "sequence_window_steps": WINDOW,
        "sequence_window_minutes": WINDOW * INTERVAL_MINUTES,
        "forecast_horizons_steps": HORIZONS,
        "forecast_horizons_minutes": [h * INTERVAL_MINUTES for h in HORIZONS],
        "sampling_interval_minutes": INTERVAL_MINUTES,
    },
    "suitability_score_formula": {
        "temp_score / ph_score": "3 if value in optimal_range; 2 if value in operational_range (but not optimal); 1 otherwise",
        "do_score": "3 if do >= do_opt_threshold; 2 if do_ops_threshold <= do < do_opt_threshold; 1 if do < do_ops_threshold",
        "pond_score": "mean(temp_score, ph_score, do_score)",
        "do_gap_ops": "do - do_ops_threshold  (negative => below operational limit)",
        "do_gap_opt": "do - do_opt_threshold  (negative => below optimal limit)",
        "rolling suitability features": "rolling mean/sum of the above scores over the trailing W in {4,8,24} timesteps, min_periods=W (no forward-looking values; first W-1 rows of each pond series are NaN and dropped)",
    },
    "excursion_formula": {
        "do_excursion_flag": "1 if do < do_ops_threshold else 0",
        "do_warning_flag": "1 if do_ops_threshold <= do < do_opt_threshold else 0",
        "do_excursion_duration_steps": "running count of consecutive prior timesteps (including current) with do_excursion_flag == 1; resets to 0 on the first non-excursion step",
        "do_excursion_duration_minutes": "do_excursion_duration_steps * 15",
        "rolling excursion features": "rolling mean/sum of do_excursion_flag / do_warning_flag over trailing W in {4,8,24} timesteps, min_periods=W",
    },
    "stability_formula": {
        "{var}_roll_sd{W}": "rolling standard deviation of temp/ph/do over the trailing W in {4,8,24} timesteps, min_periods=W",
        "{var}_roll_cv{W}": "{var}_roll_sd{W} / {var}_roll_mean{W}  (coefficient of variation)",
    },
    "temporal_features_formula": {
        "{var}_lag{L}": "value of temp/ph/do exactly L timesteps ago, L in {1,2,3,4,8,12,24}",
        "{var}_roll_mean{W}/_roll_min{W}/_roll_max{W}": "rolling statistic of temp/ph/do over trailing W in {4,8,24} timesteps, min_periods=W",
        "hour_sin / hour_cos": "sin/cos encoding of time-of-day, period = 24h",
        "day_index": "elapsed days since the first observation of that pond's monitoring period",
        "pond_* (one-hot)": "one-hot indicator of pond identity",
    },
    "no_future_leakage_guarantee": (
        "All lag and rolling-window operations use pandas .shift()/.rolling() in the default "
        "(backward-looking, non-centered) mode with min_periods=window, computed independently "
        "per pond after chronological sorting. A feature at row t therefore uses only "
        "observations at or before t. This is verified programmatically in the "
        "'Leakage / temporal-alignment audit' cell below, which recomputes a random sample of "
        "features directly from the raw series and checks for exact agreement."
    ),
    "boundary_sample_handling": (
        "Any row where a lag/rolling feature is undefined (the first max(LAGS, WINDOWS)-1 "
        "timesteps of each pond) is NaN. A training/eval sequence of length WINDOW is discarded "
        "entirely if it contains ANY NaN across its window or in its target horizons "
        "(see make_sequences()). This is why the number of valid sequences per pond "
        "(~1,293) is slightly lower than the naive count 1344-WINDOW-max(HORIZONS)+1=1317 "
        "reported in the previous manuscript draft; the revised manuscript text should use "
        "the corrected count."
    ),
}
with open(f'{RESULT_DIR}/feature_engineering_config.json', 'w') as f:
    json.dump(feature_engineering_config, f, indent=2)

print('\nSaved feature_engineering_config.json with explicit formulas for every feature group.')


## 7. Sequence Builder + Rolling-Origin Walk-Forward Folds (Reviewer 2 #4)

In [ ]:
# =========================
# 7. Sequence builder and evaluation folds
#    (original 70/15/15 split + rolling-origin walk-forward folds)
# =========================
def make_sequences(data, feature_cols, window=WINDOW, horizons=HORIZONS):
    Xs, Ys, metas, last_dos = [], [], [], []
    max_h = max(horizons)
    for pond, g in data.groupby('pond'):
        g = g.sort_values('timestamp').reset_index(drop=True)
        feature_values = g[feature_cols].values.astype(float)
        do_values = g['do'].values.astype(float)
        timestamps = g['timestamp'].values
        ops_thr = float(RANGES[pond]['do_ops'])
        for end in range(window - 1, len(g) - max_h):
            start = end - window + 1
            X = feature_values[start:end + 1]
            y = np.array([do_values[end + h] for h in horizons], dtype=float)
            if np.isnan(X).any() or np.isnan(y).any():
                continue
            Xs.append(X)
            Ys.append(y)
            last_dos.append(do_values[end])
            metas.append({
                'pond': pond,
                'end_timestamp': pd.Timestamp(timestamps[end]),
                'do_ops_thr': ops_thr,
                'last_do': do_values[end],
            })
    X = np.array(Xs, dtype=float)
    Y = np.array(Ys, dtype=float)
    meta = pd.DataFrame(metas)
    last_do = np.array(last_dos, dtype=float)
    return X, Y, meta, last_do

# ---------------------------------------------------------------------
# IMPORTANT SIMPLIFICATION (verified against the real dataset before
# relying on it): every feature set in FEATURE_SETS below shares the same
# NaN warm-up boundary, because TEMPORAL_FEATURES alone already requires a
# 24-step lag/rolling window -- the same requirement suitability/excursion/
# stability features impose. As a result, make_sequences() returns EXACTLY
# the same (pond, end_timestamp) rows, in the same order, no matter which
# feature_cols is passed in (checked with meta_raw.equals(meta_full) on the
# actual iot_data.csv -- both give 5,172 rows, identical row-for-row). We
# therefore compute the split/fold POSITION INDICES only once and reuse
# them for every scenario's X/Y array.
# ---------------------------------------------------------------------

def original_chronological_split(meta, train_ratio=0.70, val_ratio=0.15):
    """The single 70/15/15 chronological split used in the previous manuscript
    draft. Kept as fold 0 so results remain comparable to earlier numbers."""
    train_idx, val_idx, test_idx = [], [], []
    for pond, g in meta.groupby('pond'):
        idx = g.sort_values('end_timestamp').index.to_numpy()
        n = len(idx)
        n_train = int(np.floor(train_ratio * n))
        n_val = int(np.floor(val_ratio * n))
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:n_train + n_val])
        test_idx.extend(idx[n_train + n_val:])
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

def make_walkforward_folds(meta, n_blocks=N_BLOCKS):
    """NEW -- rolling-origin (walk-forward) folds requested by Reviewer 2 #4.
    Each pond's sequences are cut chronologically into `n_blocks` equal-size,
    non-overlapping blocks. Each fold trains on an EXPANDING prefix of
    blocks, validates on the next block, and tests on the block after that,
    so every fold's test period is a genuinely different, later slice of
    the calendar than fold 0's test period, per pond."""
    meta = meta.copy()
    meta['block'] = -1
    for pond, g in meta.groupby('pond'):
        order = g.sort_values('end_timestamp').index
        n = len(order)
        block_id = np.minimum((np.floor(np.arange(n) / n * n_blocks)).astype(int), n_blocks - 1)
        meta.loc[order, 'block'] = block_id

    fold_specs = []
    for k in range(n_blocks - 2):
        train_blocks = list(range(0, k + 2))
        val_block = k + 2
        test_block = k + 3
        if test_block >= n_blocks:
            break
        fold_specs.append({'fold': k + 1, 'train_blocks': train_blocks, 'val_block': val_block, 'test_block': test_block})
    return meta, fold_specs

def scale_3d(X_train, X_val, X_test):
    scaler = StandardScaler()
    n_train, t, f = X_train.shape
    scaler.fit(X_train.reshape(-1, f))
    def transform(X):
        n, t, f = X.shape
        return scaler.transform(X.reshape(-1, f)).reshape(n, t, f)
    return transform(X_train), transform(X_val), transform(X_test), scaler

# Feature sets -- Raw_GRU and Full_Assessment_GRU now mirror their LSTM
# counterparts EXACTLY, fixing the confounded comparison (Reviewer 2 #2).
FEATURE_SETS = {
    'Raw_LSTM': BASE_FEATURES + TEMPORAL_FEATURES,
    'Raw_GRU': BASE_FEATURES + TEMPORAL_FEATURES,                                            # NEW -- was missing before
    'LSTM_Suitability': BASE_FEATURES + TEMPORAL_FEATURES + SUITABILITY_FEATURES,
    'LSTM_Excursion': BASE_FEATURES + TEMPORAL_FEATURES + EXCURSION_FEATURES,
    'LSTM_Stability': BASE_FEATURES + TEMPORAL_FEATURES + STABILITY_FEATURES,
    'Full_Assessment_Guided_LSTM': BASE_FEATURES + TEMPORAL_FEATURES + SUITABILITY_FEATURES + EXCURSION_FEATURES + STABILITY_FEATURES,
    'Full_Assessment_GRU': BASE_FEATURES + TEMPORAL_FEATURES + SUITABILITY_FEATURES + EXCURSION_FEATURES + STABILITY_FEATURES,  # RENAMED from GRU_Baseline
    # NEW -- reimplemented 2024+ literature-style baselines (Reviewer 1 #8,
    # Reviewer 2 #1), trained on the SAME Raw feature set as Raw_LSTM/Raw_GRU
    # for a controlled comparison.
    'GRU_NBeats_lite': BASE_FEATURES + TEMPORAL_FEATURES,
    'CNN_GRU_Attention': BASE_FEATURES + TEMPORAL_FEATURES,
}
FEATURE_SETS = {k: uniq(v) for k, v in FEATURE_SETS.items()}

MODEL_FAMILY = {
    'Raw_LSTM': 'lstm', 'LSTM_Suitability': 'lstm', 'LSTM_Excursion': 'lstm',
    'LSTM_Stability': 'lstm', 'Full_Assessment_Guided_LSTM': 'lstm',
    'Raw_GRU': 'gru', 'Full_Assessment_GRU': 'gru',
    'GRU_NBeats_lite': 'gru_nbeats', 'CNN_GRU_Attention': 'cnn_gru_attention',
}

# Build sequences once (any feature set gives identical meta -- see note above)
X_full, Y_full, meta_full, last_do_full = make_sequences(df, FEATURE_SETS['Full_Assessment_GRU'])
orig_train_idx, orig_val_idx, orig_test_idx = original_chronological_split(meta_full)
meta_with_blocks, fold_specs = make_walkforward_folds(meta_full, n_blocks=N_BLOCKS)

# Precompute the (train_idx, val_idx, test_idx) position arrays for every
# fold ONCE. These same position arrays are valid for every scenario's own
# X/Y array because all scenarios share the identical row order (verified above).
FOLD_INDEX_MAP = {0: (orig_train_idx, orig_val_idx, orig_test_idx)}
for spec in fold_specs:
    tr = meta_with_blocks.index[meta_with_blocks['block'].isin(spec['train_blocks'])].to_numpy()
    va = meta_with_blocks.index[meta_with_blocks['block'] == spec['val_block']].to_numpy()
    te = meta_with_blocks.index[meta_with_blocks['block'] == spec['test_block']].to_numpy()
    FOLD_INDEX_MAP[spec['fold']] = (tr, va, te)

split_summary = []
for fold_id, (tr, va, te) in FOLD_INDEX_MAP.items():
    for split_name, idxs in [('train', tr), ('validation', va), ('test', te)]:
        s = meta_full.loc[idxs].groupby('pond').size().reset_index(name='n_sequences')
        s['split'] = split_name
        s['fold'] = fold_id
        split_summary.append(s)
split_summary = pd.concat(split_summary, ignore_index=True)
split_summary.to_csv(f'{RESULT_DIR}/sequence_split_summary_all_folds.csv', index=False)

# also save the exact timestamps in each fold's test set, per pond, for reproducibility
for fold_id, (tr, va, te) in FOLD_INDEX_MAP.items():
    meta_full.loc[te, ['pond', 'end_timestamp']].to_csv(f'{RESULT_DIR}/fold{fold_id}_test_timestamps.csv', index=False)

FOLD_IDS = FOLD_IDS_QUICK if QUICK_TEST_MODE else FOLD_IDS_FULL

print('Full sequence shape:', X_full.shape, Y_full.shape)
print('Fold specs (rolling-origin, walk-forward):')
for spec in fold_specs:
    print(' ', spec)
print('Folds that will actually be run this session (FOLD_IDS):', FOLD_IDS)
display(split_summary.pivot_table(index=['fold', 'pond'], columns='split', values='n_sequences'))


## 8. Leakage / Temporal-Alignment Audit (Reviewer 2 #3, part 1)

In [ ]:
# =========================
# 8. Leakage / temporal-alignment audit (Reviewer 2 #3, part 1)
# =========================
# Reviewer 2 asked us to "formally demonstrate that every assessment feature at
# prediction time uses only information available at or before the forecast
# origin". Rather than just asserting this in prose, we recompute a random
# sample of lag/rolling features directly from the raw per-pond series
# (using ONLY rows up to and including that timestamp) and check for exact
# agreement with the stored feature column. Any mismatch would indicate a
# forward-looking bug.

def audit_no_future_leakage(data, feature_cols, n_checks=300, seed=0):
    rng = np.random.default_rng(seed)
    mismatches = []
    checked = 0
    for pond, g in data.groupby('pond'):
        g = g.sort_values('timestamp').reset_index(drop=True)
        n = len(g)
        if n <= 30:
            continue
        sample_idx = rng.choice(np.arange(30, n), size=min(n_checks, n - 30), replace=False)
        for i in sample_idx:
            past_slice = g.iloc[:i + 1]
            for col in feature_cols:
                if '_lag' in col and col.split('_lag')[0] in ['temp', 'ph', 'do']:
                    base, lag = col.rsplit('_lag', 1)
                    lag = int(lag)
                    expected = past_slice[base].iloc[-1 - lag] if len(past_slice) > lag else np.nan
                    got = g.loc[i, col]
                    if pd.notna(expected) and pd.notna(got) and not np.isclose(expected, got):
                        mismatches.append((pond, i, col, expected, got))
                elif '_roll_mean' in col and col.split('_roll_mean')[0] in ['temp', 'ph', 'do']:
                    base, win = col.split('_roll_mean')
                    win = int(win)
                    expected = past_slice[base].iloc[-win:].mean() if len(past_slice) >= win else np.nan
                    got = g.loc[i, col]
                    if pd.notna(expected) and pd.notna(got) and not np.isclose(expected, got):
                        mismatches.append((pond, i, col, expected, got))
            checked += 1
    return checked, mismatches

audit_features = [c for c in (TEMPORAL_FEATURES + SUITABILITY_FEATURES + STABILITY_FEATURES)
                   if ('_lag' in c or '_roll_mean' in c) and c.split('_lag')[0].split('_roll_mean')[0] in ['temp', 'ph', 'do']]
n_checked, mismatches = audit_no_future_leakage(df, audit_features, n_checks=200)

audit_result = {
    'n_feature_columns_audited': len(audit_features),
    'n_row_checks_performed': n_checked,
    'n_mismatches_found': len(mismatches),
    'conclusion': 'PASS - no forward-looking feature detected' if len(mismatches) == 0 else 'FAIL - see mismatches',
}
with open(f'{RESULT_DIR}/leakage_audit_result.json', 'w') as f:
    json.dump(audit_result, f, indent=2, default=str)

print(audit_result)
if mismatches:
    print('First mismatches:', mismatches[:10])

# NOTE on the *other* half of Reviewer 2 #3 (circularity between suitability
# features and the low-DO label definition, as opposed to strict future
# leakage): that is addressed separately below via the threshold-sensitivity
# and threshold-independent AUC analysis, after model predictions are available.


## 9. PatchTST-Lite Builder — Fixed Comparator Architecture

This is the only trainable architecture introduced in this notebook. It is a compact patch-based Transformer comparator inspired by PatchTST and sized for the short 24-step input window.

The architecture is **fixed** here:
- patch length = 4
- d_model = 64
- heads = 4
- feed-forward dimension = 128
- dropout = 0.20
- Adam learning rate = 0.001

These values are not used to retune any previously reported model.


In [ ]:
class PatchPositionalEmbedding(layers.Layer):
    """Learnable per-patch positional embedding, added via a trainable weight
    of shape (1, n_patches, d_model) rather than an Embedding-lookup-on-a-
    constant-range-tensor. The Embedding+tf.range approach produced a graph
    with an implicitly fixed batch size baked into the positional term,
    which raised an InvalidArgumentError ('Incompatible shapes: [32,4] vs.
    [6,4]') on the final, smaller batch of an epoch under XLA compilation on
    GPU. A weight with a leading singleton batch dimension broadcasts
    correctly against any batch size via standard tensor addition."""
    def __init__(self, n_patches, d_model, **kwargs):
        super().__init__(**kwargs)
        self.n_patches = n_patches
        self.d_model = d_model

    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            name='pos_embedding',
            shape=(1, self.n_patches, self.d_model),
            initializer='random_normal',
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        return x + self.pos_emb

    def get_config(self):
        config = super().get_config()
        config.update({'n_patches': self.n_patches, 'd_model': self.d_model})
        return config


def build_patchtst_lite(input_shape, output_dim=4, patch_len=4, d_model=64, num_heads=4,
                         ff_dim=128, dropout=0.2, lr=1e-3):
    window, n_features = input_shape
    assert window % patch_len == 0, f'window ({window}) must be divisible by patch_len ({patch_len})'
    n_patches = window // patch_len

    inputs = layers.Input(shape=input_shape)

    # --- Patchify: (window, n_features) -> (n_patches, patch_len * n_features) ---
    x = layers.Reshape((n_patches, patch_len * n_features))(inputs)

    # --- Linear patch embedding ---
    x = layers.Dense(d_model, name='patch_embedding')(x)

    # --- Learnable positional embedding, added to every patch token ---
    x = PatchPositionalEmbedding(n_patches, d_model, name='add_positional')(x)

    # --- Single Transformer encoder block (multi-head self-attention + FFN) ---
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=max(d_model // num_heads, 1), dropout=dropout, name='self_attention'
    )(x, x)
    x = layers.Add(name='attn_residual')([x, attn_out])
    x = layers.LayerNormalization(epsilon=1e-6, name='attn_norm')(x)

    ffn = layers.Dense(ff_dim, activation='relu', name='ffn_dense1')(x)
    ffn = layers.Dense(d_model, name='ffn_dense2')(ffn)
    x = layers.Add(name='ffn_residual')([x, ffn])
    x = layers.LayerNormalization(epsilon=1e-6, name='ffn_norm')(x)

    # --- Pool across patches and forecast ---
    x = layers.GlobalAveragePooling1D(name='patch_pooling')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(output_dim)(x)

    model = models.Model(inputs, out, name='PatchTST_lite')
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

PATCHTST_CONFIG = {
    'patch_len': 4,
    'd_model': 64,
    'num_heads': 4,
    'ff_dim': 128,
    'dropout': 0.20,
    'lr': 1e-3,
}

print('PatchTST-lite fixed config:', PATCHTST_CONFIG)


## 10. Evaluation Metrics — Same Definitions as Original Notebook


In [ ]:
# =========================
# 10. Evaluation metric functions
#     regression / low-DO (timestamp) / decision-level: unchanged from the
#     previous notebook. NEW: event-based detection metrics (Reviewer 2 #6),
#     threshold-sensitivity and threshold-independent AUC analysis
#     (Reviewer 2 #3, part 2).
# =========================

def regression_metrics(y_true, y_pred, model_name):
    rows = []
    for j, h in enumerate(HORIZONS):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        mae = mean_absolute_error(yt, yp)
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mape = np.mean(np.abs((yt - yp) / np.where(yt == 0, np.nan, yt))) * 100
        r2 = r2_score(yt, yp)
        rows.append({'model': model_name, 'horizon_step': h, 'horizon': HORIZON_LABELS[h], 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2})
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    rows.append({'model': model_name, 'horizon_step': 'overall', 'horizon': 'overall',
                 'MAE': mean_absolute_error(yt, yp), 'RMSE': np.sqrt(mean_squared_error(yt, yp)),
                 'MAPE': np.mean(np.abs((yt - yp) / np.where(yt == 0, np.nan, yt))) * 100,
                 'R2': r2_score(yt, yp)})
    return pd.DataFrame(rows)

def low_do_metrics(y_true, y_pred, meta_test, model_name):
    rows = []
    thr = meta_test['do_ops_thr'].values.astype(float)
    for j, h in enumerate(HORIZONS):
        actual_low = (y_true[:, j] < thr).astype(int)
        pred_low = (y_pred[:, j] < thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(actual_low, pred_low, labels=[0, 1]).ravel()
        rows.append({
            'model': model_name, 'horizon_step': h, 'horizon': HORIZON_LABELS[h],
            'actual_low': int(actual_low.sum()), 'pred_low': int(pred_low.sum()),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'precision': precision_score(actual_low, pred_low, zero_division=0),
            'recall': recall_score(actual_low, pred_low, zero_division=0),
            'f1_score': f1_score(actual_low, pred_low, zero_division=0)
        })
    return pd.DataFrame(rows)

def decision_level_metrics(y_true, y_pred, meta_test, model_name):
    thr = meta_test['do_ops_thr'].values.astype(float)
    actual_low = (y_true < thr[:, None]).astype(int)
    pred_low = (y_pred < thr[:, None]).astype(int)
    actual_any = actual_low.any(axis=1).astype(int)
    pred_any = pred_low.any(axis=1).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual_any, pred_any, labels=[0, 1]).ravel()
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    missed_alarm_rate = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    decision_accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan
    return pd.DataFrame([{
        'model': model_name, 'TP_event': int(tp), 'FP_event': int(fp), 'FN_event': int(fn), 'TN_event': int(tn),
        'false_alarm_rate': false_alarm_rate, 'missed_alarm_rate': missed_alarm_rate, 'decision_accuracy': decision_accuracy,
    }])

# ---- NEW: event-based detection (Reviewer 2 #6) --------------------------
def _group_events(binary_flags):
    events = []
    start = None
    for i, v in enumerate(binary_flags):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1))
            start = None
    if start is not None:
        events.append((start, len(binary_flags) - 1))
    return events

def event_based_metrics(y_true, y_pred, meta_test, model_name):
    """Groups timestamp-level low-DO flags into contiguous EPISODES per pond
    (sorted chronologically) and reports event-level recall / false-alarm
    counts, in addition to the timestamp-level metrics already computed by
    low_do_metrics(). This distinguishes 'how many distinct low-DO episodes
    did we catch' from 'how many individual timestamps did we catch',
    which Reviewer 2 explicitly asked for."""
    thr = meta_test['do_ops_thr'].values.astype(float)
    rows = []
    for j, h in enumerate(HORIZONS):
        actual_low_all = (y_true[:, j] < thr).astype(int)
        pred_low_all = (y_pred[:, j] < thr).astype(int)
        per_pond_rows = []
        for pond in meta_test['pond'].unique():
            mask = (meta_test['pond'] == pond).values
            order = np.argsort(meta_test.loc[mask, 'end_timestamp'].values)
            a = actual_low_all[mask][order]
            p = pred_low_all[mask][order]
            actual_events = _group_events(a)
            pred_events = _group_events(p)
            detected = sum(1 for (s, e) in actual_events if p[s:e + 1].any())
            def overlaps_any_actual(ps, pe, events=actual_events):
                return any(not (pe < s or ps > e) for (s, e) in events)
            false_alarm_events = sum(1 for (ps, pe) in pred_events if not overlaps_any_actual(ps, pe))
            per_pond_rows.append({
                'pond': pond, 'n_actual_events': len(actual_events), 'n_detected_events': detected,
                'n_pred_events': len(pred_events), 'n_false_alarm_events': false_alarm_events,
            })
        pp = pd.DataFrame(per_pond_rows)
        total_actual = pp['n_actual_events'].sum()
        total_detected = pp['n_detected_events'].sum()
        rows.append({
            'model': model_name, 'horizon_step': h, 'horizon': HORIZON_LABELS[h],
            'n_actual_events_total': int(total_actual), 'n_detected_events_total': int(total_detected),
            'event_recall': (total_detected / total_actual) if total_actual > 0 else np.nan,
            'n_pred_events_total': int(pp['n_pred_events'].sum()),
            'n_false_alarm_events_total': int(pp['n_false_alarm_events'].sum()),
        })
    return pd.DataFrame(rows)

# ---- NEW: threshold-sensitivity + threshold-independent AUC (Reviewer 2 #3) ----
def threshold_sensitivity(y_true, y_pred, meta_test, model_name, scales=(0.8, 0.9, 1.0, 1.1, 1.2)):
    """Recomputes low-DO recall using the operational threshold scaled by
    +/-20%. If a model's apparent recall advantage collapses once the
    threshold is perturbed away from the exact value baked into the
    suitability features, that is evidence the advantage is partly an
    artifact of feature/label circularity rather than a genuinely robust
    precursor signal."""
    base_thr = meta_test['do_ops_thr'].values.astype(float)
    rows = []
    for j, h in enumerate(HORIZONS):
        for scale in scales:
            thr = base_thr * scale
            actual_low = (y_true[:, j] < thr).astype(int)
            pred_low = (y_pred[:, j] < thr).astype(int)
            rows.append({
                'model': model_name, 'horizon': HORIZON_LABELS[h], 'threshold_scale': scale,
                'recall': recall_score(actual_low, pred_low, zero_division=0),
                'precision': precision_score(actual_low, pred_low, zero_division=0),
                'n_positive': int(actual_low.sum()),
            })
    return pd.DataFrame(rows)

def threshold_free_auc(y_true, y_pred, meta_test, model_name):
    """ROC-AUC / PR-AUC of the CONTINUOUS predicted DO value as a risk score
    against the binary low-DO label, at each horizon. This ranks models
    without committing to any single decision threshold, directly answering
    Reviewer 2's request for a threshold-independent sensitivity check."""
    thr = meta_test['do_ops_thr'].values.astype(float)
    rows = []
    for j, h in enumerate(HORIZONS):
        actual_low = (y_true[:, j] < thr).astype(int)
        risk_score = -y_pred[:, j]
        if actual_low.sum() == 0 or actual_low.sum() == len(actual_low):
            roc_auc, pr_auc = np.nan, np.nan
        else:
            roc_auc = roc_auc_score(actual_low, risk_score)
            pr_auc = average_precision_score(actual_low, risk_score)
        rows.append({'model': model_name, 'horizon': HORIZON_LABELS[h], 'roc_auc': roc_auc, 'pr_auc': pr_auc})
    return pd.DataFrame(rows)

print('Metric functions ready (regression, low-DO, decision-level, event-based, threshold-sensitivity, threshold-free AUC).')


## 11. PatchTST-Lite Only Training

This cell deliberately trains **only PatchTST-lite**.

It uses `FEATURE_SETS['Raw_LSTM']`, i.e. the same Raw sensor + temporal feature set used by the original Raw LSTM/Raw GRU comparison. The cell asserts that the feature count is 58 before training.

The original fold policy is preserved:
- Fold 0: all five seeds.
- Folds 1–2: seed 42 only.


In [ ]:

MULTISEED_FOLDS = [0]
PRIMARY_SEED = SEEDS[0]
PRIMARY_FOLD = 0

PATCH_SCENARIO = 'PatchTST_lite'
PATCH_FEATURES = FEATURE_SETS['Raw_LSTM']

print('PatchTST input feature count:', len(PATCH_FEATURES))
assert len(PATCH_FEATURES) == 58, (
    f'Expected 58 Raw sensor+temporal features, found {len(PATCH_FEATURES)}. '
    'Stop here rather than changing the comparison protocol.'
)

# Build PatchTST sequences from the frozen Raw feature set.
X_patch, Y_patch, meta_patch, last_do_patch = make_sequences(df, PATCH_FEATURES)

# Verify exact sequence alignment with the original fold index reference.
assert len(meta_patch) == len(meta_full), 'PatchTST sequence count differs from frozen reference.'
assert meta_patch[['pond','end_timestamp']].reset_index(drop=True).equals(
    meta_full[['pond','end_timestamp']].reset_index(drop=True)
), 'PatchTST sequence ordering differs from frozen reference.'

all_reg, all_low, all_dec, all_event = [], [], [], []
all_thr_sensitivity, all_auc = [], []
run_log = []

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

for fold_id in FOLD_IDS:
    seeds_this_fold = SEEDS if fold_id in MULTISEED_FOLDS else [PRIMARY_SEED]
    train_idx, val_idx, test_idx = FOLD_INDEX_MAP[fold_id]

    X_train, X_val, X_test = X_patch[train_idx], X_patch[val_idx], X_patch[test_idx]
    y_train, y_val, y_test = Y_patch[train_idx], Y_patch[val_idx], Y_patch[test_idx]
    meta_test = meta_patch.loc[test_idx].reset_index(drop=True)

    for seed in seeds_this_fold:
        t0 = time.time()
        set_all_seeds(seed)

        # Fit scaler on training data only, exactly as in the original notebook.
        X_train_s, X_val_s, X_test_s, scaler = scale_3d(X_train, X_val, X_test)

        model = build_patchtst_lite(
            input_shape=X_train_s.shape[1:],
            output_dim=len(HORIZONS),
            **PATCHTST_CONFIG
        )

        cb = [
            callbacks.EarlyStopping(
                monitor='val_loss',
                patience=PATIENCE,
                restore_best_weights=True
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                patience=max(2, PATIENCE // 2),
                factor=0.5,
                min_lr=1e-5
            )
        ]

        history = model.fit(
            X_train_s, y_train,
            validation_data=(X_val_s, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            callbacks=cb
        )

        y_pred = model.predict(X_test_s, verbose=0)
        tag = f'{PATCH_SCENARIO}__fold{fold_id}__seed{seed}'

        all_reg.append(regression_metrics(y_test, y_pred, tag))
        all_low.append(low_do_metrics(y_test, y_pred, meta_test, tag))
        all_dec.append(decision_level_metrics(y_test, y_pred, meta_test, tag))

        # Detailed primary-run diagnostics use the same rule as the old notebook.
        if fold_id == PRIMARY_FOLD and seed == PRIMARY_SEED:
            all_event.append(event_based_metrics(y_test, y_pred, meta_test, tag))
            all_thr_sensitivity.append(threshold_sensitivity(y_test, y_pred, meta_test, tag))
            all_auc.append(threshold_free_auc(y_test, y_pred, meta_test, tag))

            pd.DataFrame({
                'pond': meta_test['pond'].values,
                'end_timestamp': meta_test['end_timestamp'].values,
                'do_ops_thr': meta_test['do_ops_thr'].values,
                **{f'y_true_h{h}': y_test[:, j] for j, h in enumerate(HORIZONS)},
                **{f'y_pred_h{h}': y_pred[:, j] for j, h in enumerate(HORIZONS)},
            }).to_csv(
                f'{RESULT_DIR}/prediction_audit_{PATCH_SCENARIO}_fold{fold_id}.csv',
                index=False
            )

        run_log.append({
            'scenario': PATCH_SCENARIO,
            'family': 'patchtst_lite',
            'fold': fold_id,
            'seed': seed,
            'n_features': len(PATCH_FEATURES),
            'n_train': len(train_idx),
            'n_val': len(val_idx),
            'n_test': len(test_idx),
            'epochs_ran': len(history.history['loss']),
            'best_val_loss': float(np.min(history.history['val_loss'])),
            'train_seconds': round(time.time() - t0, 1),
        })
        print(f'[fold {fold_id} | seed {seed}] PatchTST_lite done in {time.time()-t0:.1f}s')

regression_df = pd.concat(all_reg, ignore_index=True)
lowdo_df = pd.concat(all_low, ignore_index=True)
decision_df = pd.concat(all_dec, ignore_index=True)
event_df = pd.concat(all_event, ignore_index=True) if all_event else pd.DataFrame()
threshold_sensitivity_df = pd.concat(all_thr_sensitivity, ignore_index=True) if all_thr_sensitivity else pd.DataFrame()
auc_df = pd.concat(all_auc, ignore_index=True) if all_auc else pd.DataFrame()
run_log_df = pd.DataFrame(run_log)

for _df in [regression_df, lowdo_df, decision_df]:
    parts = _df['model'].str.extract(r'^(?P<scenario>.+)__fold(?P<fold>\d+)__seed(?P<seed>NA|\d+)$')
    _df['scenario'] = parts['scenario']
    _df['fold'] = parts['fold'].astype(int)
    _df['seed'] = parts['seed']

for _df in [event_df, threshold_sensitivity_df, auc_df]:
    if len(_df):
        parts = _df['model'].str.extract(r'^(?P<scenario>.+)__fold(?P<fold>\d+)__seed(?P<seed>NA|\d+)$')
        _df['scenario'] = parts['scenario']
        _df['fold'] = parts['fold'].astype(int)
        _df['seed'] = parts['seed']

regression_df.to_csv(f'{RESULT_DIR}/patchtst_regression_metrics_all_runs.csv', index=False)
lowdo_df.to_csv(f'{RESULT_DIR}/patchtst_low_do_metrics_all_runs.csv', index=False)
decision_df.to_csv(f'{RESULT_DIR}/patchtst_decision_level_metrics_all_runs.csv', index=False)
run_log_df.to_csv(f'{RESULT_DIR}/patchtst_run_log.csv', index=False)

if len(event_df):
    event_df.to_csv(f'{RESULT_DIR}/patchtst_event_based_metrics_primary_run.csv', index=False)
if len(threshold_sensitivity_df):
    threshold_sensitivity_df.to_csv(f'{RESULT_DIR}/patchtst_threshold_sensitivity_primary_run.csv', index=False)
if len(auc_df):
    auc_df.to_csv(f'{RESULT_DIR}/patchtst_threshold_free_auc_primary_run.csv', index=False)

print('\nPatchTST-only training complete.')
display(run_log_df)


## 12. PatchTST-Lite Uncertainty, Rolling-Origin Summary, and Editor-Ready Outputs

Fold-0 uncertainty is computed across the five seeds. Rolling-origin folds remain single-seed, exactly matching the frozen manuscript protocol.

This notebook does **not** recompute statistical tests against the old models, because doing that correctly requires the original per-seed result files. After this run, keep this PatchTST package separate; we can merge it with the frozen old result package and calculate matched comparisons without changing any legacy number.


In [ ]:

def mean_sd_ci(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = x.mean()
    sd = x.std(ddof=1) if n > 1 else 0.0
    if n > 1:
        ci = stats.t.interval(confidence, n - 1, loc=mean, scale=stats.sem(x))
    else:
        ci = (mean, mean)
    return mean, sd, ci[0], ci[1]

def aggregate_with_uncertainty(df_metric, value_col, group_cols, fold_for_uncertainty=0):
    sub = df_metric[df_metric['fold'] == fold_for_uncertainty].copy()
    rows = []
    for keys, g in sub.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        mean, sd, lo, hi = mean_sd_ci(g[value_col].values)
        row = dict(zip(group_cols, keys))
        row.update({
            f'{value_col}_mean': mean,
            f'{value_col}_sd': sd,
            f'{value_col}_ci_low': lo,
            f'{value_col}_ci_high': hi,
            'n_seeds': len(g)
        })
        rows.append(row)
    return pd.DataFrame(rows)

rmse_uncertainty = aggregate_with_uncertainty(
    regression_df[regression_df['horizon'] != 'overall'],
    'RMSE', ['scenario', 'horizon']
)
recall_uncertainty = aggregate_with_uncertainty(
    lowdo_df, 'recall', ['scenario', 'horizon']
)
decision_acc_uncertainty = aggregate_with_uncertainty(
    decision_df, 'decision_accuracy', ['scenario']
)

rmse_uncertainty.to_csv(f'{RESULT_DIR}/patchtst_rmse_uncertainty_fold0_multiseed.csv', index=False)
recall_uncertainty.to_csv(f'{RESULT_DIR}/patchtst_recall_uncertainty_fold0_multiseed.csv', index=False)
decision_acc_uncertainty.to_csv(f'{RESULT_DIR}/patchtst_decision_accuracy_uncertainty_fold0_multiseed.csv', index=False)

# Rolling-origin RMSE
robustness_rows = []
for h in HORIZON_LABELS.values():
    sub = regression_df[regression_df['horizon'] == h]
    per_fold = sub.groupby('fold')['RMSE'].mean()
    robustness_rows.append({
        'scenario': PATCH_SCENARIO,
        'horizon': h,
        **{f'RMSE_fold{k}': v for k, v in per_fold.items()}
    })
robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(f'{RESULT_DIR}/patchtst_rmse_across_rolling_origin_folds.csv', index=False)

# Editor-friendly fold-0 summaries
fold0_reg = regression_df[(regression_df['fold'] == 0) & (regression_df['horizon'] != 'overall')]
fold0_low = lowdo_df[lowdo_df['fold'] == 0]
fold0_dec = decision_df[decision_df['fold'] == 0]

editor_summary = {
    'PatchTST_lite_feature_count': int(len(PATCH_FEATURES)),
    'PatchTST_lite_overall_RMSE_mean_across_horizons_and_seeds':
        float(fold0_reg['RMSE'].mean()),
    'PatchTST_lite_mean_recall_across_horizons_and_seeds':
        float(fold0_low['recall'].mean()),
    'PatchTST_lite_mean_false_alarm_rate_across_seeds':
        float(fold0_dec['false_alarm_rate'].mean()),
    'PatchTST_lite_mean_missed_alarm_rate_across_seeds':
        float(fold0_dec['missed_alarm_rate'].mean()),
    'PatchTST_lite_mean_decision_accuracy_across_seeds':
        float(fold0_dec['decision_accuracy'].mean()),
}

if len(auc_df):
    editor_summary['PatchTST_lite_mean_ROC_AUC_primary_seed'] = float(auc_df['roc_auc'].mean())
    editor_summary['PatchTST_lite_mean_PR_AUC_primary_seed'] = float(auc_df['pr_auc'].mean())

with open(f'{RESULT_DIR}/patchtst_editor_summary.json', 'w') as f:
    json.dump(editor_summary, f, indent=2)

print('PatchTST RMSE uncertainty, fold 0:')
display(rmse_uncertainty)
print('PatchTST recall uncertainty, fold 0:')
display(recall_uncertainty)
print('PatchTST decision accuracy uncertainty, fold 0:')
display(decision_acc_uncertainty)
print('PatchTST rolling-origin RMSE:')
display(robustness_df)
print('Editor-ready summary:')
print(json.dumps(editor_summary, indent=2))


## 13. Reproducibility Manifest and ZIP Export


In [ ]:

import platform

manifest = {
    'purpose': 'PatchTST-lite-only add-on experiment for IJIES second-round revision',
    'legacy_models_retrained': False,
    'legacy_hyperparameter_search_rerun': False,
    'quick_test_mode': QUICK_TEST_MODE,
    'seeds': SEEDS,
    'multiseed_folds': [0],
    'rolling_origin_single_seed_folds': [1, 2],
    'window_steps': WINDOW,
    'forecast_horizons_steps': HORIZONS,
    'sampling_interval_minutes': INTERVAL_MINUTES,
    'patchtst_feature_set': 'Raw_LSTM feature set (sensor + temporal only)',
    'patchtst_n_features': len(PATCH_FEATURES),
    'patchtst_config': PATCHTST_CONFIG,
    'epochs_max': EPOCHS,
    'batch_size': BATCH_SIZE,
    'early_stopping_patience': PATIENCE,
    'tensorflow_version': tf.__version__,
    'python_version': platform.python_version(),
}

with open(f'{RESULT_DIR}/patchtst_frozen_protocol_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)

readme = """# PatchTST-Lite Add-On Results — Frozen Original Protocol

This package contains ONLY the new PatchTST-lite comparator requested in the
second-round IJIES review.

The previously reported LSTM/GRU/GRU-N-Beats/CNN-GRU-Attention results were
NOT retrained and MUST remain frozen when updating the manuscript.

PatchTST-lite:
- uses exactly the Raw sensor + temporal feature set (58 features)
- uses the same 24-step history and 15/30/45/60-min forecast horizons
- uses the same chronological split and rolling-origin fold indices
- uses training-only StandardScaler
- fold 0: five seeds (42, 7, 123, 2024, 99)
- folds 1-2: seed 42 only
- maximum 120 epochs, batch 32, early stopping patience 15 in the full run

Merge these PatchTST result CSVs with the frozen legacy result package only
AFTER training is complete. Do not replace or recalculate legacy model rows.
"""
with open(f'{RESULT_DIR}/README_PATCHTST_ONLY.txt', 'w') as f:
    f.write(readme)

zip_path = 'patchtst_only_frozen_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(RESULT_DIR):
        for file in files:
            path = os.path.join(root, file)
            zf.write(path, arcname=os.path.relpath(path, RESULT_DIR))

print('ZIP created:', zip_path)
print('Files:')
for f_ in sorted(os.listdir(RESULT_DIR)):
    print(' -', f_)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Download manually:', zip_path)
